# Memory and Summarization

Agents that run for any length of time accumulate context: every method call adds events, every tool return adds more, and the prompt going into the LLM keeps growing until it is no longer affordable to send. NOOA gives you two orthogonal tools to deal with this: `TokenBudgetSummarizer` compresses old turns in place so the running conversation stays small, and `nooa_memory` persists facts across sessions so an agent can pick up where a previous instance left off. This notebook demonstrates both.

## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. Replace `"your-api-key"` with a real key for hosted providers; local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
# model = get_llm_client("openai/openai/openai/gpt-5.5", api_key="your-api-key", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)


## A `ResearchAssistant` that accumulates context

We'll start with an agent whose job is to read short documents one at a time and then, on demand, produce notes over everything it has read. Each `read` call adds another chunk to the event history; each `summarize_field` call has to look back over all of it. This is the classic long-context shape.

In [ ]:
from nooa import Agent
from nooa.agentdoc import spec


class ResearchAssistant(Agent, llm=model):
    """You read short research documents and take notes on them."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Expose the event log so we can inspect it from outside the agent.
        spec(self, "events", hidden=False)

    async def read(self, doc: str) -> str:
        """Read the document and produce a one-sentence gist of it.

        The document itself will remain in your event history so you can
        refer back to it in later calls.
        """
        ...

    async def summarize_field(self) -> str:
        """Summarize the overall field of research covered by every document
        you have read so far. Be concise: two or three sentences.
        """
        ...

Now give it a handful of short documents to read. We'll watch the event tape grow with each call.

In [ ]:
DOCS = [
    "Recent work on retrieval-augmented generation (RAG) shows that combining vector search with dense re-ranking substantially improves answer grounding on long-tail queries.",
    "Sparse mixture-of-experts models activate only a fraction of parameters per token, which reduces inference cost while preserving model quality on standard benchmarks.",
    "Speculative decoding uses a small draft model to propose several tokens ahead, which are then verified in parallel by the target model, cutting latency significantly.",
    "KV-cache compression schemes such as low-rank projection and quantization reduce the memory footprint of long-context inference without meaningful accuracy loss.",
    "Constitutional AI methods train a model to critique and revise its own outputs against a written set of principles, reducing the need for human preference labels.",
    "Structured decoding constrains model outputs to a formal grammar, guaranteeing valid JSON or SQL without post-hoc parsing and retries.",
]

In [ ]:
agent = ResearchAssistant()

for i, doc in enumerate(DOCS, 1):
    await agent.read(doc)
    n_events = len(agent.events.query())
    print(f"after doc {i}: {n_events} events")

print()
print(await agent.summarize_field())

The event count climbs monotonically. Each `read` adds a task, the LLM's reasoning, and the return; nothing is ever thrown away. Fine at six documents, ruinous at six hundred: the prompt going into the LLM on the final `summarize_field` call contains every earlier task, every earlier gist, and every earlier document verbatim.

## `TokenBudgetSummarizer`: compress old turns in place

The fix is to compress older regions of the event tape into short summaries once the running prompt crosses a token budget. `TokenBudgetSummarizer` does exactly this: it subscribes to the agent's event manager, waits until the actual prompt-token count reported by the provider exceeds `max_tokens`, then collapses the oldest events into a single `Summary` event while leaving the most recent `preserve_recent` turns untouched.

Install it with `TokenBudgetSummarizer.install(agent, config=...)`. The threshold below is intentionally very low so that the collapse triggers within a handful of `read` calls.

In [ ]:
from nooa.agents import TokenBudgetSummarizer
from nooa.config import TokenBudgetConfig

agent = ResearchAssistant()
TokenBudgetSummarizer.install(
    agent,
    config=TokenBudgetConfig(
        max_tokens=1500,     # aggressively low, purely for demo purposes
        preserve_recent=2,   # keep the last two turns verbatim
        target_chars=400,    # ask the summary itself to stay ~400 chars
    ),
)

for i, doc in enumerate(DOCS, 1):
    await agent.read(doc)
    n_events = len(agent.events.query())
    print(f"after doc {i}: {n_events} events")

Two things worth noticing in the counts. The tape no longer grows without bound: once the running prompt crosses 1500 tokens the summarizer schedules a collapse in the background, and the next turn opens with a single `Summary` event replacing the older range. The two most recent turns are always preserved verbatim, which keeps the model's local reasoning intact.

In [ ]:
# Inspect what the tape looks like after compression.
for tag in agent.event_manager.keys():
    evt = agent.event_manager[tag]
    kind = type(evt).__name__
    preview = str(evt)[:80].replace("\n", " ")
    print(f"  [{tag:>3}] {kind:<20} {preview}")

In [ ]:
# The agent still knows what it read, but the older documents now live
# inside a compressed summary rather than as verbatim events.
print(await agent.summarize_field())

The summary is lossy by construction. The framework tells the summarizer to keep decisions, exact numbers, and technical terms while compressing discussions and dropping process chatter. In exchange, you get an agent whose prompt stays bounded no matter how long it runs.

> **Takeaway.** `TokenBudgetSummarizer` is fire-and-forget context compression. Install it once, pick a token ceiling, and the running prompt stops growing.

## `nooa_memory`: persist facts across sessions

Summarization keeps a single run bounded, but it does nothing for the next run: the moment you throw the agent object away, the compressed history goes with it. A separate concern, and a separate subpackage, handles cross-session persistence.

`nooa_memory` gives the agent a small database of facts it can write to (`remember`) and query (`recall`). The store is backed by SQLite and lives at a path you choose, so a fresh agent instance in a later session can read what an earlier instance wrote. You opt in by mixing in `MemoryToolsMixin` and installing a `MemoryManager`.

In [ ]:
import tempfile, os

from nooa_memory import MemoryConfig, MemoryManager, MemoryToolsMixin, MemoryType

# Persistent DB path shared across both agent instances below.
MEMORY_PATH = os.path.join(tempfile.gettempdir(), "nooa_research_memory.db")
if os.path.exists(MEMORY_PATH):
    os.remove(MEMORY_PATH)


class ResearchAssistantWithMemory(MemoryToolsMixin, Agent, llm=model):
    """A research assistant that can commit facts to long-term memory."""

    async def read(self, doc: str) -> str:
        """Read the document and produce a one-sentence gist. If the document
        contains a stable fact worth carrying across sessions, call
        self.remember(fact, type=\"info\") on it before returning.
        """
        ...

### Session A: write

The first agent instance reads a couple of documents, notes what matters, and shuts down. We can either let the agent decide what to remember via its `remember` tool during a read, or write facts directly through the manager. Both are shown below.

In [ ]:
agent_a = ResearchAssistantWithMemory()
mgr_a = MemoryManager.install(
    agent_a,
    config=MemoryConfig(enabled=True, path=MEMORY_PATH),
)

# Write a couple of facts directly through the manager.
mgr_a.remember("The project's default vector index is NumpyVectorIndex.", type=MemoryType.INFO)
mgr_a.remember("KV-cache compression targets long-context inference.", type=MemoryType.INFO)

print("memories stored:", mgr_a.store.count())

# Tear the first session down.
mgr_a.uninstall()
del agent_a, mgr_a

### Session B: recall

Now build a completely fresh agent, point it at the same DB path, and query. Nothing about `agent_a` is in memory; the only channel between the two is the file on disk.

In [ ]:
agent_b = ResearchAssistantWithMemory()
mgr_b = MemoryManager.install(
    agent_b,
    config=MemoryConfig(enabled=True, path=MEMORY_PATH),
)

print("memories visible to session B:", mgr_b.store.count())
print()

# Deliberate recall via the manager (no LLM involved).
for m in mgr_b.recall("which vector index does the project default to?", k=2):
    print(f"  [{m.type}] {m.content}")

The agent-facing side of the same lookup lives on the mixin: `agent_b.recall(query, k=...)` returns the same `Memory` objects, and because `remember` and `recall` are ordinary methods on the class, the LLM can call them itself during a task without any tool registration.

In [ ]:
# Same recall, but through the agent's public surface — this is what the
# LLM would use if it decided to consult memory mid-task.
for m in agent_b.recall("which vector index does the project default to?", k=2):
    print(f"  [{m.type}] {m.content}")

## Trade-offs

Summarization and memory look superficially similar — both trim the amount of raw history the model sees — but they solve different problems and have different failure modes. Summarization is lossy compression of the current run: cheap to enable, keeps the prompt bounded, and dependent on the summarizer's LLM producing faithful summaries (exact numbers or rare terms can silently disappear). Memory is durable, structured storage across runs: it costs a disk write and a similarity query, it requires you to decide what is worth remembering, and it introduces a second retrieval failure mode when the query at recall time doesn't match how the fact was phrased at write time. In practice the two are complementary: summarization keeps a single agent affordable to run for hours, and memory lets the next agent start already knowing what the previous one learned.

> **Takeaway.** Summarization compresses *this* conversation; memory persists facts to the *next* one. Reach for the first when the running prompt is too big, and for the second when a fresh agent needs to inherit what an earlier one learned.